# 财务风险预警 · 探索性数据分析（EDA）

本 Notebook 用于论文实证章节的前置探索，覆盖：样本分布、特征分布、单变量区分度（IV）、
特征间相关性、以及一个快速基线模型的 ROC/KS 表现。

**前置条件**：已执行 `python -m etl.run_pipeline`。

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from config import settings
from config.feature_registry import FEATURE_KEYS, TARGET_COLUMN
from etl.io_utils import read_table

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

panel = read_table(settings.FEATURE_MATRIX)
print(f"样本：{panel.shape[0]} 行 × {panel.shape[1]} 列")
panel.head()

## 1. 样本结构与标签分布

检查年度、行业、正负样本比例，确认是否存在年份或行业的结构性断层。

In [ ]:
summary = panel.groupby("year").agg(观测数=("stock_code", "count"), 企业数=("stock_code", "nunique"), ST率=(TARGET_COLUMN, "mean"))
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
summary["ST率"].plot(marker="o", ax=axes[0], title="年度 ST 比例", ylabel="比例")
panel["industry"].value_counts().plot(kind="barh", ax=axes[1], title="行业观测数")
plt.tight_layout()

## 2. 单变量区分度（IV 值）

IV 值用于第一步特征筛选：IV<0.02 视为无预测力。这里只在训练窗口内估计，避免使用未来信息。

In [ ]:
from etl.stats_utils import calculate_iv

train = panel[panel["year"].between(*settings.TRAIN_YEAR_RANGE)]
features = [c for c in FEATURE_KEYS if c in train.columns]
iv = calculate_iv(train[features], train[TARGET_COLUMN])
display(iv.head(20))

iv.head(20).iloc[::-1].plot(kind="barh", x="feature", y="iv", figsize=(7, 6), legend=False, title="IV TOP20")

## 3. 正负样本的特征差异（标准化均值差）

均值差/标准差用于直观展示哪些指标在企业被 ST 前已显著恶化，可作为论文的描述性统计证据。

In [ ]:
pos, neg = train[train[TARGET_COLUMN] == 1][features], train[train[TARGET_COLUMN] == 0][features]
effect = ((pos.mean() - neg.mean()) / neg.std()).sort_values(key=abs, ascending=False)
effect.head(15).to_frame("标准化均值差")

## 4. 快速基线：逻辑回归 vs XGBoost

仅用于探索，正式结论以 `models/train.py` 的五模型对比与 OOT 验证为准。

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from models.metrics import evaluate_model

X_tr, y_tr = train[features], train[TARGET_COLUMN]
valid = panel[panel["year"] == settings.VALID_YEAR]
X_va, y_va = valid[features], valid[TARGET_COLUMN]

models = {
    "Logistic": Pipeline([("s", StandardScaler()), ("m", LogisticRegression(max_iter=2000, class_weight="balanced"))]),
    "XGBoost": XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05, eval_metric="auc", random_state=42),
}

plt.figure(figsize=(5, 4.5))
for name, model in models.items():
    model.fit(X_tr, y_tr)
    prob = model.predict_proba(X_va)[:, 1]
    m = evaluate_model(y_va, prob, name=name)
    print(f"{name}: AUC={m['auc']:.4f} KS={m['ks']:.4f} F1={m['f1']:.4f}")
    fpr, tpr, _ = roc_curve(y_va, prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC={m['auc']:.3f})")

plt.plot([0, 1], [0, 1], "k--", linewidth=1)
plt.xlabel("假阳性率"); plt.ylabel("真阳性率"); plt.legend(); plt.title("验证集 ROC")
plt.tight_layout()

## 5. 小结

- 样本量与正样本比例符合设计预期；
- 偿债能力、现金流类特征的 IV 值最高，与文献结论一致；
- 树模型相对线性基线有明显提升，支持选用 XGBoost 作为主模型；
- 正式结论需结合 `models/artifacts/evaluation_report.md` 的 OOT 结果。